In [0]:
df = spark.read.table("customer_churn_dataset_testing_master")
df.show()

+----------+---+------+------+---------------+-------------+-------------+-----------------+---------------+-----------+----------------+-----+
|CustomerID|Age|Gender|Tenure|Usage Frequency|Support Calls|Payment Delay|Subscription Type|Contract Length|Total Spend|Last Interaction|Churn|
+----------+---+------+------+---------------+-------------+-------------+-----------------+---------------+-----------+----------------+-----+
|         1| 22|Female|    25|             14|            4|           27|            Basic|        Monthly|        598|               9|    1|
|         2| 41|Female|    28|             28|            7|           13|         Standard|        Monthly|        584|              20|    0|
|         3| 47|  Male|    27|             10|            2|           29|          Premium|         Annual|        757|              21|    0|
|         4| 35|  Male|     9|             12|            5|           17|          Premium|      Quarterly|        232|              18

In [0]:
df.count()

64374

In [0]:
display(df)

CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
1,22,Female,25,14,4,27,Basic,Monthly,598,9,1
2,41,Female,28,28,7,13,Standard,Monthly,584,20,0
3,47,Male,27,10,2,29,Premium,Annual,757,21,0
4,35,Male,9,12,5,17,Premium,Quarterly,232,18,0
5,53,Female,58,24,9,2,Standard,Annual,533,18,0
6,30,Male,41,14,10,10,Premium,Monthly,500,29,0
7,47,Female,37,15,9,28,Basic,Quarterly,574,14,1
8,54,Female,36,11,0,18,Standard,Monthly,323,16,0
9,36,Male,20,5,10,8,Basic,Monthly,687,8,0
10,65,Male,8,4,2,23,Basic,Annual,995,10,0


In [0]:
df.groupBy("Churn").count().show()

+-----+-----+
|Churn|count|
+-----+-----+
|    1|30493|
|    0|33881|
+-----+-----+



In [0]:
df.describe().show()

+-------+------------------+------------------+------+------------------+------------------+-----------------+-----------------+-----------------+---------------+-----------------+------------------+-------------------+
|summary|        CustomerID|               Age|Gender|            Tenure|   Usage Frequency|    Support Calls|    Payment Delay|Subscription Type|Contract Length|      Total Spend|  Last Interaction|              Churn|
+-------+------------------+------------------+------+------------------+------------------+-----------------+-----------------+-----------------+---------------+-----------------+------------------+-------------------+
|  count|             64374|             64374| 64374|             64374|             64374|            64374|            64374|            64374|          64374|            64374|             64374|              64374|
|   mean|           32187.5| 41.97098207350794|  NULL|31.994827104110357|15.080234256066113|5.400689719451953|17.1339515

In [0]:
df = df.dropna()

In [0]:
from pyspark.sql.functions import col

df = df.withColumn("Age", col("Age").cast("int"))
df = df.withColumn("Total Spend", col("Total Spend").cast("double"))
df = df.withColumn("Churn", col("Churn").cast("int"))

In [0]:
from pyspark.sql.functions import when

df = df.withColumn(
    "high_support",
    when(col("Support Calls") > 3, 1).otherwise(0)
)

In [0]:
from pyspark.sql.functions import when

df = df.withColumn(
    "high_support",
    when(col("Support Calls") > 3, 1).otherwise(0)
)

In [0]:
df = df.withColumn(
    "high_delay",
    when(col("Payment Delay") > 5, 1).otherwise(0)
)

In [0]:
df.groupBy("Churn").count().show()

+-----+-----+
|Churn|count|
+-----+-----+
|    1|30493|
|    0|33881|
+-----+-----+



In [0]:
df.groupBy("Subscription Type", "Churn").count().show()

+-----------------+-----+-----+
|Subscription Type|Churn|count|
+-----------------+-----+-----+
|            Basic|    1|10356|
|         Standard|    0|11325|
|          Premium|    0|11461|
|            Basic|    0|11095|
|         Standard|    1|10177|
|          Premium|    1| 9960|
+-----------------+-----+-----+



In [0]:
df.groupBy("Support Calls", "Churn").count().show()

+-------------+-----+-----+
|Support Calls|Churn|count|
+-------------+-----+-----+
|            4|    1| 1639|
|            7|    0| 2522|
|            2|    0| 3698|
|            5|    0| 2632|
|            9|    0| 2605|
|           10|    0| 2589|
|            9|    1| 4059|
|            0|    0| 3764|
|            7|    1| 4077|
|            6|    0| 2615|
|            4|    0| 3509|
|            1|    0| 3801|
|            3|    1| 1173|
|            8|    0| 2596|
|            8|    1| 4053|
|            2|    1| 1114|
|            3|    0| 3550|
|           10|    1| 3998|
|            5|    1| 4025|
|            6|    1| 4024|
+-------------+-----+-----+
only showing top 20 rows


In [0]:
df.groupBy("Contract Length", "Churn").count().show()

+---------------+-----+-----+
|Contract Length|Churn|count|
+---------------+-----+-----+
|        Monthly|    1|11421|
|        Monthly|    0|10709|
|         Annual|    0|11515|
|      Quarterly|    0|11657|
|      Quarterly|    1| 9177|
|         Annual|    1| 9895|
+---------------+-----+-----+



In [0]:
df.groupBy("Churn").avg("Total Spend").show()

+-----+-----------------+
|Churn| avg(Total Spend)|
+-----+-----------------+
|    1|519.3361427212803|
|    0|560.5419556683687|
+-----+-----------------+



In [0]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression

# =========================
# 1. CHARGEMENT DATA
# =========================
df = spark.read.table("customer_churn_dataset_testing_master")

# =========================
# 2. ENCODING VARIABLES CATEGORIELLES
# =========================

# Gender
gender_indexer = StringIndexer(inputCol="Gender", outputCol="GenderIndex")
df = gender_indexer.fit(df).transform(df)

# Subscription Type
sub_indexer = StringIndexer(inputCol="Subscription Type", outputCol="SubscriptionIndex")
df = sub_indexer.fit(df).transform(df)

# Contract Length
contract_indexer = StringIndexer(inputCol="Contract Length", outputCol="ContractIndex")
df = contract_indexer.fit(df).transform(df)

# =========================
# 3. FEATURES SELECTION
# =========================

features = [
    "Age",
    "Tenure",
    "Usage Frequency",
    "Support Calls",
    "Payment Delay",
    "Total Spend",
    "Last Interaction",
    "GenderIndex",
    "SubscriptionIndex",
    "ContractIndex"
]

assembler = VectorAssembler(
    inputCols=features,
    outputCol="features"
)

df_ml = assembler.transform(df)

# =========================
# 4. TRAIN / TEST SPLIT
# =========================

train, test = df_ml.randomSplit([0.8, 0.2], seed=42)

# =========================
# 5. MODEL TRAINING
# =========================

lr = LogisticRegression(
    featuresCol="features",
    labelCol="Churn"
)

model = lr.fit(train)

# =========================
# 6. PREDICTION
# =========================

predictions = model.transform(test)

predictions.select("features", "Churn", "prediction", "probability").show(5)

# =========================
# 7. EVALUATION (OPTIONNEL)
# =========================

from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="Churn",
    rawPredictionCol="rawPrediction"
)

accuracy = evaluator.evaluate(predictions)
print("AUC Score:", accuracy)

+--------------------+-----+----------+--------------------+
|            features|Churn|prediction|         probability|
+--------------------+-----+----------+--------------------+
|[47.0,27.0,10.0,2...|    0|       1.0|[0.39444728013527...|
|[47.0,37.0,15.0,9...|    1|       1.0|[0.02335151089124...|
|[36.0,20.0,5.0,10...|    0|       0.0|[0.77372597110810...|
|[42.0,46.0,27.0,5...|    0|       0.0|[0.96860113398850...|
|[62.0,39.0,19.0,2...|    0|       0.0|[0.88816755965406...|
+--------------------+-----+----------+--------------------+
only showing top 5 rows
AUC Score: 0.9021241843384714


In [0]:
train, test = df_ml.randomSplit([0.8, 0.2])

In [0]:
predictions = model.transform(test)
predictions.select("Churn", "prediction").show()

+-----+----------+
|Churn|prediction|
+-----+----------+
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    1|       1.0|
|    1|       0.0|
|    0|       0.0|
|    0|       1.0|
|    0|       0.0|
|    0|       1.0|
|    0|       0.0|
|    0|       0.0|
|    1|       1.0|
|    0|       1.0|
|    0|       1.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
+-----+----------+
only showing top 20 rows


In [0]:
df_final = predictions.select(
    "CustomerID",
    "Age",
    "Churn",
    "prediction"
)

In [0]:
df_final.write.mode("overwrite").saveAsTable("churn_dashboard")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5736030635935923>, line 1
----> 1 df_final.write.mode("overwrite").saveAsTable("churn_dashboard")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1536     req.user_context.user_id = self._user_id
   1537 self._set_command_in_plan(req.plan, command)
-> 153

In [0]:
df.createOrReplaceTempView("churn_view")

In [0]:
%sql
SELECT Churn, COUNT(*) as total
FROM churn_view
GROUP BY Churn

Churn,total
1,30493
0,33881


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT `Subscription Type`, Churn, COUNT(*) as total
FROM churn_view
GROUP BY `Subscription Type`, Churn

Subscription Type,Churn,total
Basic,1,10356
Standard,0,11325
Premium,0,11461
Basic,0,11095
Standard,1,10177
Premium,1,9960


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT `Support Calls`, Churn, COUNT(*) as total
FROM churn_view
GROUP BY `Support Calls`, Churn

Support Calls,Churn,total
4,1,1639
7,0,2522
2,0,3698
5,0,2632
9,0,2605
10,0,2589
9,1,4059
0,0,3764
7,1,4077
6,0,2615


Databricks visualization. Run in Databricks to view.

In [0]:
import numpy as np
import pandas as pd

In [0]:
def predict_churn(age, tenure, usage, support_calls, payment_delay, total_spend):

    data = np.array([[age, tenure, usage,
                      support_calls, payment_delay,
                      total_spend]])

    pred = model.predict(data)[0]
    proba = model.predict_proba(data)[0][1]

    return pred, proba

In [0]:
pip install gradio

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import gradio as gr
import numpy as np

def predict_churn(age, gender, tenure, usage, last_interaction,
                  support_calls, payment_delay,
                  total_spend, subscription_type, contract_length):

    #  encoding
    gender = 1 if gender.lower() == "male" else 0

    sub_map = {"basic":0, "standard":1, "premium":2}
    contract_map = {"monthly":0, "yearly":1}

    subscription_type = sub_map[subscription_type.lower()]
    contract_length = contract_map[contract_length.lower()]

    #  input model
    data = np.array([[
        age, gender, tenure, usage, last_interaction,
        support_calls, payment_delay,
        total_spend, subscription_type, contract_length
    ]])

    pred = model.predict(data)[0]
    proba = model.predict_proba(data)[0][1]

    if pred == 1:
        return f" HIGH RISK ({round(proba*100,2)}%)"
    else:
        return f" LOW RISK ({round(proba*100,2)}%)"

In [0]:
interface = gr.Interface(
    fn=predict_churn,
    
    inputs=[
        gr.Number(label="Age"),
        gr.Radio(["Male", "Female"], label="Gender"),
        gr.Number(label="Tenure"),
        gr.Number(label="Usage Frequency"),
        gr.Number(label="Last Interaction"),
        gr.Number(label="Support Calls"),
        gr.Number(label="Payment Delay"),
        gr.Number(label="Total Spend"),
        gr.Dropdown(["Basic", "Standard", "Premium"]),
        gr.Dropdown(["Monthly", "Yearly"])
    ],

    outputs="text",

    title="Customer Churn Prediction System",
    description="Predict if a customer will churn using ML model"
)

In [0]:
interface.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://84b59cf211bf76785c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
